<a href="https://colab.research.google.com/github/Sangeetha3315/Agentic-AI-and-computer-vision-workshop-projects/blob/main/smart_attendance_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Attendance System — Session 2 Project
### Face Detection + Recognition + Live Headcount, built with YOLO + face embeddings

**Why this project:** it's a complete, realistic detection pipeline that a company could
actually use — and it demonstrates almost everything from Session 2 in one flow:
- Object/face detection with bounding boxes (YOLO-based face detector)
- Real-time video inference concepts (frame sampling, live webcam loop)
- A practical postprocessing layer: turning raw detections into a structured, useful output (an attendance log)

**What we'll build:**
1. **Enrollment phase** — capture a few photos of each person and compute a "face signature" (embedding) for each
2. **Live detection phase** — point the webcam at a person, detect their face, match it against enrolled faces, and log attendance with a timestamp
3. **Dashboard** — a simple live headcount + attendance table, exportable as CSV

**Runtime:** Runtime → Change runtime type → GPU (recommended, but CPU works fine too since it's just face-sized inference).


## Step 0 — Setup

In [ ]:
# Install necessary libraries quietly (-q suppresses output)
!pip install -q ultralytics deepface opencv-python-headless pandas

# Import OpenCV library for image processing
import cv2
# Import NumPy for numerical operations, especially with arrays
import numpy as np
# Import Pandas for data manipulation and analysis, particularly with DataFrames
import pandas as pd
# Import os module for interacting with the operating system (e.g., file paths)
import os
# Import time module for time-related tasks (e.g., delays)
import time
# Import datetime class from datetime module for working with dates and times
from datetime import datetime
# Import display, Javascript, HTML for displaying rich output in Colab
from IPython.display import display, Javascript, HTML
# Import eval_js to execute JavaScript code and get its result
from google.colab.output import eval_js
# Import b64decode for decoding base64 encoded strings (used for image data)
from base64 import b64decode
# Import YOLO class from ultralytics for object detection
from ultralytics import YOLO
# Import DeepFace for face recognition and analysis
from deepface import DeepFace
# Import matplotlib.pyplot for plotting and visualization
import matplotlib.pyplot as plt

# Print a message indicating that the setup is complete
print("Setup complete.")

## Step 1 — Webcam Capture Utility

Same reusable browser-webcam bridge used in Session 1 — opens your camera, lets you click
**Capture**, and saves the frame as a file we can process in Python.


In [ ]:
# Define a function to capture a photo from the webcam
def take_photo(filename='photo.jpg', quality=0.8):
    # JavaScript code to interact with the browser's webcam
    js = Javascript('''
    async function takePhoto(quality) {
      // Create a div element to contain video and capture button
      const div = document.createElement('div');
      // Create a capture button
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      // Append the capture button to the div
      div.appendChild(capture);

      // Create a video element
      const video = document.createElement('video');
      video.style.display = 'block';
      // Request access to the user's media devices (webcam)
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      // Append the div to the document body
      document.body.appendChild(div);
      // Append the video element to the div
      div.appendChild(video);
      // Set the video source to the webcam stream
      video.srcObject = stream;
      // Play the video stream
      await video.play();

      // Set the iframe height to show the video feed properly
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      // Wait for the capture button to be clicked
      await new Promise((resolve) => capture.onclick = resolve);

      // Create a canvas element to draw the video frame
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      // Draw the current video frame onto the canvas
      canvas.getContext('2d').drawImage(video, 0, 0);
      // Stop the video stream tracks
      stream.getVideoTracks()[0].stop();
      // Remove the div containing video and button from the document
      div.remove();
      // Return the image data as a base64 encoded JPEG string
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    # Display the JavaScript code in the Colab output
    display(js)
    # Execute the JavaScript function and get the image data
    data = eval_js('takePhoto({})'.format(quality))
    # Decode the base64 image data (remove the 'data:image/jpeg;base64,' prefix)
    binary = b64decode(data.split(',')[1])
    # Open the specified filename in binary write mode
    with open(filename, 'wb') as f:
        # Write the binary image data to the file
        f.write(binary)
    # Return the filename
    return filename

# Print a message indicating that webcam capture is ready
print("Webcam capture ready.")

## Step 2 — Load a Face Detector (YOLO-based)

We use a YOLOv8 model fine-tuned for face detection. This gives us fast bounding boxes
around faces — the exact "detection fundamentals" (boxes, confidence, NMS handled internally
by the model) from the lecture, applied to a real task.


In [ ]:
# ---- Step 2 (rectified): Load a Face Detector, with a reliable fallback ----
# The community YOLO-face release URLs on GitHub are occasionally flaky/renamed,
# which causes a corrupted/empty .pt download. This cell verifies the download
# and automatically falls back to OpenCV's DNN face detector (very stable,
# no external repo dependency) if the YOLO weights fail to load — so the demo
# never breaks live.

# Import os module for interacting with the operating system (e.g., file paths)
import os
# Import OpenCV library for image processing
import cv2
# Import NumPy for numerical operations, especially with arrays
import numpy as np
# Import matplotlib.pyplot for plotting and visualization
import matplotlib.pyplot as plt

# Initialize face_backend to None, it will be 'yolo' or 'opencv_dnn'
face_backend = None

# ---- Attempt 1: YOLOv8 face weights ----
# List of URLs to try for downloading YOLOv8 face detection weights
yolo_urls = [
    "https://github.com/derronqi/yolov8-face/releases/download/v0.0.0/yolov8n-face.pt",
    "https://github.com/akanametov/yolov8-face/releases/download/v0.0.0/yolov8n-face.pt",
]

# Loop through each URL to attempt downloading and loading YOLO weights
for url in yolo_urls:
    # Use wget to download the file quietly and save it as 'yolov8n-face.pt'
    os.system(f"wget -q '{url}' -O yolov8n-face.pt")
    # Check if the file exists and its size is greater than 1MB (to avoid corrupted downloads)
    if os.path.exists("yolov8n-face.pt") and os.path.getsize("yolov8n-face.pt") > 1_000_000:
        try:
            # Import YOLO class from ultralytics (if not already imported)
            from ultralytics import YOLO
            # Load the YOLOv8 face model
            face_detector = YOLO("yolov8n-face.pt")
            # Set the backend to 'yolo'
            face_backend = "yolo"
            # Print success message
            print(f"Loaded YOLO face detector from: {url}")
            # Break the loop if successful
            break
        except Exception as e:
            # Handle exceptions during loading (e.g., corrupted file)
            print(f"Downloaded but failed to load from {url}: {e}")
    else:
        # Print message if download failed or file was too small
        print(f"Download failed or file too small from: {url}")

# ---- Fallback: OpenCV DNN face detector (stable, no flaky repo dependency) ----
# If YOLO backend was not successfully loaded
if face_backend is None:
    # Inform about falling back to OpenCV DNN
    print("Falling back to OpenCV DNN face detector...")
    # Download the prototxt (model architecture) file
    os.system("wget -q https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt -O deploy.prototxt")
    # Download the caffemodel (trained weights) file
    os.system("wget -q https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel -O res10_300x300.caffemodel")
    # Load the pre-trained Caffe model for face detection
    opencv_face_net = cv2.dnn.readNetFromCaffe("deploy.prototxt", "res10_300x300.caffemodel")
    # Set the backend to 'opencv_dnn'
    face_backend = "opencv_dnn"
    # Print success message
    print("OpenCV DNN face detector ready.")

# Print the active face detection backend
print(f"\nActive face detection backend: {face_backend}")

# ---- Unified detect_faces() so the rest of the notebook doesn't need to change ----
# Define a unified function to detect faces regardless of the backend
def detect_faces(image_path, conf=0.5):
    # Read the image from the specified path
    img = cv2.imread(image_path)
    # Get the height and width of the image
    h, w = img.shape[:2]

    # If the YOLO backend is active
    if face_backend == "yolo":
        # Perform prediction using the YOLO face detector
        results = face_detector.predict(image_path, conf=conf, verbose=False)
        # Extract bounding boxes from the results, convert to numpy array
        boxes = results[0].boxes.xyxy.cpu().numpy() if len(results[0].boxes) > 0 else np.array([])
        return boxes

    # If the OpenCV DNN backend is active
    else:  # opencv_dnn
        # Create a blob from the image: resize, normalize, mean subtraction
        blob = cv2.dnn.blobFromImage(cv2.resize(img, (300, 300)), 1.0,
                                       (300, 300), (104.0, 177.0, 123.0))
        # Set the input for the neural network
        opencv_face_net.setInput(blob)
        # Perform a forward pass to get detections
        detections = opencv_face_net.forward()
        # Initialize an empty list to store bounding boxes
        boxes = []
        # Iterate over each detection
        for i in range(detections.shape[2]):
            # Get the confidence score for the current detection
            score = detections[0, 0, i, 2]
            # If the confidence score is above the threshold
            if score > conf:
                # Get the bounding box coordinates and scale them to the original image size
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                # Append the box to the list
                boxes.append(box)
        # Return the list of bounding boxes as a numpy array
        return np.array(boxes)

# ---- Quick test ----
# Take a photo to test the face detector
test_path = take_photo('detector_test.jpg')
# Detect faces in the captured photo
boxes = detect_faces(test_path)
# Read the image again and convert from BGR to RGB for matplotlib display
img = cv2.cvtColor(cv2.imread(test_path), cv2.COLOR_BGR2RGB)
# Display the image
plt.imshow(img)
# Get the current axes
ax = plt.gca()
# Iterate over each detected bounding box
for (x1, y1, x2, y2) in boxes:
    # Add a rectangle patch for each face to the plot
    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='lime', linewidth=2))
# Turn off axis labels and ticks
plt.axis('off')
# Set the title of the plot, showing the number of faces detected and the backend used
plt.title(f"Detected {len(boxes)} face(s) — backend: {face_backend}")
# Show the plot
plt.show()

## Step 3 — Enrollment: Register People for Attendance

For each person, capture a couple of reference photos. We crop the detected face and compute
a **face embedding** (a numeric "signature" vector) using DeepFace. This embedding is what
lets us later recognize *who* a detected face belongs to — detection alone only tells us
*that* a face exists, not whose it is.


In [ ]:
# Define the directory where enrolled faces will be stored
ENROLL_DIR = 'enrolled_faces'
# Create the directory if it doesn't exist (exist_ok=True prevents error if it already exists)
os.makedirs(ENROLL_DIR, exist_ok=True)

# Define a function to enroll a new person
def enroll_person(name, num_photos=2):
    # Create a person-specific directory inside ENROLL_DIR
    person_dir = os.path.join(ENROLL_DIR, name)
    os.makedirs(person_dir, exist_ok=True)
    # Print instructions for enrollment
    print(f"Enrolling '{name}' — capture {num_photos} clear photos of their face.")
    # Loop to capture the specified number of photos
    for i in range(num_photos):
        # Take a photo and save it with a unique name in the person's directory
        path = take_photo(f'{person_dir}/{i}.jpg')
        # Detect faces in the captured photo
        boxes = detect_faces(path)
        # If no face is found
        if len(boxes) == 0:
            # Inform the user and prompt for a retake
            print(f"  No face found in photo {i+1}, please retake — re-running capture.")
            # Retake the photo
            path = take_photo(f'{person_dir}/{i}.jpg')
            # Detect faces again in the retaken photo
            boxes = detect_faces(path)
        # Extract the coordinates of the first detected face and convert to integers
        x1, y1, x2, y2 = boxes[0].astype(int)
        # Read the captured image
        img = cv2.imread(path)
        # Crop the image to just the detected face area
        # Use max(0,...) to prevent negative coordinates if bounding box goes out of bounds
        face_crop = img[max(0,y1):y2, max(0,x1):x2]
        # Save the cropped face image back to the person's directory
        cv2.imwrite(f'{person_dir}/{i}.jpg', face_crop)
        # Confirm photo capture
        print(f"  Captured photo {i+1}/{num_photos} for {name}")
    # Confirm enrollment completion
    print(f"Enrollment complete for '{name}'.\n")

# ---- Run this once per person you want to enroll ----
# Call the enrollment function for 'Person_1' with 2 photos
# Change the name and rerun this cell for additional people
enroll_person("Person_2", num_photos=2)

**Repeat Step 3** for each additional person (change the name, re-run the cell).

In [ ]:
# Get a list of subdirectories (person names) within the ENROLL_DIR
enrolled_people = os.listdir(ENROLL_DIR)
# Print the list of currently enrolled people
print("Currently enrolled:", enrolled_people)

## Step 4 — Build the Reference Face Database

We compute and store one embedding per enrolled person (averaging across their captured
photos), so live recognition just needs to compare a new face embedding against this small
database — this is the "postprocessing / matching" step of our pipeline.


In [ ]:
# Define a function to build the face database from enrolled faces
def build_face_database(enroll_dir):
    # Initialize an empty dictionary to store the face database
    database = {}
    # Iterate through each person's directory in the enrollment directory
    for person in os.listdir(enroll_dir):
        # Construct the full path to the person's directory
        person_dir = os.path.join(enroll_dir, person)
        # Initialize a list to store embeddings for the current person
        embeddings = []
        # Iterate through each photo file in the person's directory
        for fname in os.listdir(person_dir):
            # Construct the full path to the photo file
            fpath = os.path.join(person_dir, fname)
            try:
                # Use DeepFace to get the face embedding for the current photo
                # model_name='Facenet' specifies the model to use for embedding extraction
                # enforce_detection=False prevents DeepFace from performing its own face detection,
                # assuming our detect_faces() function has already provided a cropped face.
                rep = DeepFace.represent(img_path=fpath, model_name='Facenet',
                                          enforce_detection=False)[0]['embedding']
                # Add the extracted embedding to the list for this person
                embeddings.append(rep)
            except Exception as e:
                # Handle cases where embedding extraction fails for a photo
                print(f"Skipping {fpath}: {e}")
        # If any embeddings were successfully generated for the person
        if embeddings:
            # Calculate the mean embedding for the person and store it in the database
            database[person] = np.mean(embeddings, axis=0)
    # Return the built face database
    return database

# Build the face database using the enrolled faces directory
face_db = build_face_database(ENROLL_DIR)
# Print the names of people for whom the face database has been built
print("Face database built for:", list(face_db.keys()))

## Step 5 — Live Attendance: Detect, Recognize, Log

This is the full pipeline in action: **capture frame → detect face(s) → compute embedding
for each → match against enrolled database → log attendance with timestamp if it's a new
entry for today.**

Run this cell repeatedly (e.g., once per person walking into "the room") to simulate live attendance.


In [ ]:
# Define the filename for the attendance log CSV
ATTENDANCE_LOG = 'attendance_log.csv'
# Check if the attendance log file does not exist
if not os.path.exists(ATTENDANCE_LOG):
    # If it doesn't exist, create a new DataFrame with specified columns
    # and save it as a CSV file with no index
    pd.DataFrame(columns=['Name', 'Date', 'Time']).to_csv(ATTENDANCE_LOG, index=False)

# Define a function to calculate cosine similarity between two vectors
def cosine_similarity(a, b):
    # Cosine similarity formula: dot product divided by the product of their magnitudes (L2 norms)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Define the main function to recognize faces and log attendance
def recognize_and_log(frame_path, threshold=0.5):
    # Detect faces in the given frame image
    boxes = detect_faces(frame_path)
    # Read the image using OpenCV
    img = cv2.imread(frame_path)
    # Convert the image from BGR (OpenCV default) to RGB (matplotlib default)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Load the attendance log into a Pandas DataFrame
    log_df = pd.read_csv(ATTENDANCE_LOG)
    # Get today's date in YYYY-MM-DD format
    today = datetime.now().strftime('%Y-%m-%d')
    # Get the current time in HH:MM:SS format
    now_time = datetime.now().strftime('%H:%M:%S')

    # Create a matplotlib figure and a single subplot for displaying the image
    fig, ax = plt.subplots(1, figsize=(8,6))
    # Display the RGB image on the subplot
    ax.imshow(img_rgb)

    # Iterate over each detected bounding box
    for (x1, y1, x2, y2) in boxes:
        # Convert bounding box coordinates to integers
        x1, y1, x2, y2 = map(int, (x1, y1, x2, y2))
        # Crop the image to the face area defined by the bounding box
        # Use max(0,...) to ensure coordinates are not negative
        crop = img[max(0,y1):y2, max(0,x1):x2]
        # Define a temporary path to save the cropped face
        crop_path = 'temp_crop.jpg'
        # Save the cropped face image
        cv2.imwrite(crop_path, crop)

        try:
            # Get the face embedding for the detected face using DeepFace
            # enforce_detection=False because we've already detected the face
            query_embedding = DeepFace.represent(img_path=crop_path, model_name='Facenet',
                                                  enforce_detection=False)[0]['embedding']
        except Exception:
            # If embedding extraction fails (e.g., no face detected in crop), skip this box
            continue

        # Initialize variables for the best match and its similarity score
        best_match, best_score = "Unknown", -1
        # Iterate through each enrolled person's name and their reference embedding in the face database
        for name, ref_embedding in face_db.items():
            # Calculate the cosine similarity between the query embedding and the reference embedding
            score = cosine_similarity(query_embedding, ref_embedding)
            # If the current score is higher than the best score found so far
            if score > best_score:
                # Update the best match and best score
                best_match, best_score = name, score

        # Determine the label based on the best score and the threshold
        label = best_match if best_score > threshold else "Unknown"
        # Set the color for the bounding box and text (lime for recognized, red for unknown)
        color = 'lime' if label != "Unknown" else 'red'

        # Add a rectangle patch to the plot for the detected face
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2))
        # Add text label (name and score) above the bounding box
        ax.text(x1, y1-10, f"{label} ({best_score:.2f})", color=color, fontsize=10, weight='bold')

        # Log attendance only once per person per day
        # Check if the person is already logged for today
        already_logged = ((log_df['Name'] == label) & (log_df['Date'] == today)).any()
        # If the face is recognized and not already logged for today
        if label != "Unknown" and not already_logged:
            # Create a new DataFrame row for the attendance record
            new_row = pd.DataFrame([[label, today, now_time]], columns=['Name','Date','Time'])
            # Concatenate the new row to the existing log_df
            log_df = pd.concat([log_df, new_row], ignore_index=True)
            # Save the updated DataFrame back to the CSV file
            log_df.to_csv(ATTENDANCE_LOG, index=False)
            # Print a confirmation message
            print(f"Logged attendance: {label} at {now_time}")
        # If the face is recognized but already logged for today
        elif label != "Unknown":
            # Print a message indicating they are already marked present
            print(f"{label} already marked present today.")
        # If the face is not recognized
        else:
            # Print a message for an unrecognized face
            print("Unrecognized face detected.")

    # Turn off axis labels and ticks for a cleaner image display
    ax.axis('off')
    # Set the title of the plot, showing the number of faces detected
    ax.set_title(f"Detected {len(boxes)} face(s)")
    # Show the plot
    plt.show()

# ---- Run this to simulate someone walking in and getting marked present ----
# Take a photo for attendance and save it
frame_path = take_photo('attendance_frame.jpg')
# Recognize faces in the captured frame and log attendance
recognize_and_log(frame_path)

## Step 6 — Live Dashboard: Headcount + Attendance Table

A simple structured view of today's attendance — exactly the kind of clean, structured
output an agent (Session 4!) could later read and act on (e.g., "email me if anyone hasn't
checked in by 9:15am").


In [ ]:
# Load the attendance log into a Pandas DataFrame
log_df = pd.read_csv(ATTENDANCE_LOG)
# Get today's date in YYYY-MM-DD format
today = datetime.now().strftime('%Y-%m-%d')
# Filter the log DataFrame to show only entries for today
today_log = log_df[log_df['Date'] == today]

# Print a header for the attendance dashboard, including today's date
print(f"=== Attendance Dashboard — {today} ===")
# Print the total number of people enrolled (from the face_db dictionary)
print(f"Total enrolled: {len(face_db)}")
# Print the number of unique people marked present today
print(f"Present today: {len(today_log)}")
# Calculate and print the names of people who are enrolled but not present today
print(f"Missing: {[p for p in face_db.keys() if p not in today_log['Name'].values]}")
# Print an empty line for formatting
print()
# Display the attendance log for today, resetting the index for cleaner output
display(today_log.reset_index(drop=True))

## Step 7 — Export Attendance Log


In [ ]:
# Import the files module from google.colab for file download functionality
from google.colab import files
# Initiate the download of the attendance log CSV file to the user's local machine
files.download(ATTENDANCE_LOG)

---
## Recap: What This Project Demonstrated

| Step | Concept from Session 2 lecture |
|---|---|
| Step 2 | Object detection fundamentals — bounding boxes, confidence, NMS (handled internally by YOLO) |
| Step 2 | YOLO as the real-time, speed-focused detector family |
| Step 4–5 | Full pipeline: detection → embedding → matching → structured postprocessing |
| Step 5 | Live/real-time inference on a webcam frame |
| Step 6 | Turning raw detections into a usable, structured business output |

**Extension ideas if you have extra time in the workshop:**
- Add a live video loop (instead of single-frame capture) with frame sampling (e.g., process every 10th frame) to simulate real-time monitoring
- Add a Slack/email alert for late arrivals (natural bridge into Session 4's agent tools)
- Swap the face detector for a lighter/faster YOLO variant and compare speed
- Add a simple Streamlit or Gradio front-end so it looks like a real attendance kiosk
